In [29]:
# Skeleton format: Google Drive mount (Colab only)
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Local Jupyter environment: Google Drive mount skipped.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# DS2 Challenge Team 1 — Final Reproduction Notebook

이 notebook은 최종 모델 **ResNet-50 multi-resolution TTA + ConvNeXt-Tiny 160px TTA**를 다른 PC에서 재학습하고 `DS2_challenge_team1_final.csv`를 생성합니다.

- 프로젝트 전체 폴더(`traffic_signs/`, `train.py`, `evaluate_resnet50_tta_ensemble.py`, `result.csv`)와 `dataset_extracted/`가 필요합니다.
- pretrained weight는 torchvision의 ImageNet-1K weight만 사용합니다.
- TEST label/reference/L1은 학습 및 hyperparameter 선택에 사용하지 않습니다.
- 깨끗한 PC에서 20개 checkpoint를 모두 학습하면 GPU에 따라 수 시간이 걸립니다.


### Import Libraries

In [30]:
from pathlib import Path
import csv, gc, hashlib, json, os, random, subprocess, sys, time, zipfile
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score, log_loss


PROJECT_ROOT = "/content/drive/MyDrive/project1_DL"
os.chdir(PROJECT_ROOT)

DATA_DIR = f"{PROJECT_ROOT}/dataset"
DATA_ZIP = f"{PROJECT_ROOT}/dataset.zip"

TEMPLATE = (
    f"{PROJECT_ROOT}/result.csv"
    if os.path.exists(f"{PROJECT_ROOT}/result.csv")
    else f"{PROJECT_ROOT}/results.csv"
)

FINAL_CSV = f"{PROJECT_ROOT}/DS2_challenge_team1_final.csv"
FINAL_WORK = f"{PROJECT_ROOT}/outputs/final_reproduction_team1"

os.makedirs(FINAL_WORK, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_WORKERS = 4

EXPECTED_MANIFEST_SHA256 = (
    '4d60834dd25d0848d19a060186d80324496f5064a4280ae9593db784b87b5478'
)


def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

print('project:', PROJECT_ROOT)
print(
    'torch:', torch.__version__,
    'cuda:', torch.cuda.is_available(),
    'device:', DEVICE
)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


assert os.path.exists(DATA_DIR) or os.path.exists(DATA_ZIP), \
    'dataset.zip 또는 압축 해제된 dataset 폴더가 필요합니다.'

assert os.path.exists(TEMPLATE), \
    'result.csv 또는 results.csv template이 필요합니다.'


project: /content/drive/MyDrive/project1_DL
torch: 2.11.0+cu128 cuda: True device: cuda
NVIDIA A100-SXM4-40GB


### Loading the Dataset

스켈레톤의 데이터 로딩 순서를 유지하되, 이미지 전체를 RAM 배열로 올리지 않고 경로 기반 Dataset을 사용합니다. 파일명의 `class_track_frame` 구조로 track-aware 5-fold를 만듭니다.

In [31]:
def dataset_ready(root):
    root = Path(root)
    for train in (root.rglob('*') if root.exists() else []):
        if train.is_dir() and train.name.lower() == 'train' and (train.parent/'Test').is_dir(): return True
    return False

def safe_unzip(zip_filename, extract_path):
    zip_filename, extract_path = Path(zip_filename), Path(extract_path)
    if dataset_ready(extract_path):
        print('dataset folder exists:', extract_path); return
    if not zip_filename.exists():
        raise FileNotFoundError(f'Dataset folder and zip are missing: {extract_path}, {zip_filename}')
    extract_path.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_filename, 'r') as archive: archive.extractall(extract_path)
    print('dataset extracted:', extract_path)

safe_unzip(DATA_ZIP, DATA_DIR)
assert dataset_ready(DATA_DIR), '압축 해제 후 Train/Test 폴더를 찾지 못했습니다.'

from run_resnet50_kfold import prepare_master_folds
from traffic_signs.dataset import discover_data_dirs, enumerate_test_samples, scan_image_metadata_cached

master_samples, fold_by_filename, manifest_sha = prepare_master_folds(
    str(DATA_DIR), FINAL_WORK, num_folds=5, split_seed=42
)
assert manifest_sha == EXPECTED_MANIFEST_SHA256, (manifest_sha, EXPECTED_MANIFEST_SHA256)
train_dir, test_dir = discover_data_dirs(DATA_DIR)
test_samples, test_problems = scan_image_metadata_cached(
    enumerate_test_samples(test_dir), test_dir.parent / '.test_image_metadata_cache.json'
)
assert not test_problems and len(master_samples) == 26010 and len(test_samples) == 8670
print('manifest:', manifest_sha, 'OOF:', len(master_samples), 'TEST:', len(test_samples))


dataset extracted: /content/drive/MyDrive/project1_DL/dataset


ModuleNotFoundError: No module named 'run_resnet50_kfold'

### Build and Train the Model
ResNet-50은 80/96/128px × 5-fold, ConvNeXt-Tiny는 160px × 5-fold입니다. 이미 checkpoint가 있으면 안전하게 재사용하고, 없으면 동일 설정으로 학습합니다.

In [ ]:
def make_spec(model, size, fold, directory, epochs, patience, batch_size):
    return {'model': model, 'size': size, 'fold': fold, 'dir': PROJECT_ROOT / directory,
            'epochs': epochs, 'patience': patience, 'batch_size': batch_size}

families = {'resnet80': [], 'resnet96': [], 'resnet128': [], 'convnext160': []}
for f in range(5):
    families['resnet80'].append(make_spec('resnet50', 80, f, f'outputs/experiments/resnet50_interpolated_80_112_5fold/size_80/fold_{f}', 30, 6, 64))
    families['resnet96'].append(make_spec('resnet50', 96, f, f'outputs/experiments/resnet50_finetune_5fold/size_96/fold_{f}', 30, 6, 48))
    families['resnet128'].append(make_spec('resnet50', 128, f, f'outputs/experiments/resnet50_finetune_5fold/size_128/fold_{f}', 30, 6, 32))
for f in (0, 1):
    families['convnext160'].append(make_spec('convnext_tiny', 160, f, f'outputs/experiments/gtsrb_improvement_20260805/screening/convnext_tiny_160_fold{f}', 5, 2, 32))
for f in (2, 3, 4):
    families['convnext160'].append(make_spec('convnext_tiny', 160, f, f'outputs/experiments/gtsrb_improvement_20260805/promotion_convnext160/fold_{f}', 5, 2, 32))

all_specs = sum(families.values(), [])
for spec in all_specs:
    checkpoint = spec['dir'] / 'best_model.pt'
    if checkpoint.exists():
        print('reuse', checkpoint)
        continue
    spec['dir'].mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, 'train.py', '--data-dir', str(DATA_DIR), '--output-dir', str(spec['dir']),
           '--model', spec['model'], '--image-size', str(spec['size']),
           '--preprocessing-mode', 'aspect_ratio_padding', '--augmentation', 'mild',
           '--normalization', 'auto', '--split-method', 'track', '--num-folds', '5',
           '--fold-index', str(spec['fold']), '--imbalance', 'none', '--pretrained',
           '--require-pretrained', '--no-small-stem', '--seed', '42', '--split-seed', '42',
           '--batch-size', str(spec['batch_size']), '--epochs', str(spec['epochs']),
           '--patience', str(spec['patience']), '--learning-rate', '0.0002',
           '--weight-decay', '0.0001', '--num-workers', str(NUM_WORKERS),
           '--device', DEVICE, '--amp']
    print('TRAIN', spec['model'], spec['size'], 'fold', spec['fold'])
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
assert all((s['dir'] / 'best_model.pt').exists() for s in all_specs)
print('checkpoints ready:', len(all_specs))


### Validate the Model

Track leakage와 pretrained weight 출처를 확인합니다.

In [ ]:
from traffic_signs.dataset import load_split_manifest

validation_names = []
for spec in all_specs:
    checkpoint = torch.load(spec['dir'] / 'best_model.pt', map_location='cpu', weights_only=False)
    assert checkpoint.get('pretrained_loaded') is True
    cfg = checkpoint['config']
    assert cfg['pretrained'] is True and cfg['split_method'] == 'track'
    rows = list(csv.DictReader((spec['dir'] / 'split_manifest.csv').open(encoding='utf-8')))
    train_tracks = {(r['label'], r['track_id']) for r in rows if r['split'] == 'train'}
    val_tracks = {(r['label'], r['track_id']) for r in rows if r['split'] == 'val'}
    assert not (train_tracks & val_tracks)
    if spec['model'] == 'convnext_tiny':
        validation_names.extend(r['filename'] for r in rows if r['split'] == 'val')
assert len(validation_names) == 26010 and len(set(validation_names)) == 26010
print('pretrained and track leakage checks passed')


### Test
Identity, -4°, +4°, brightness 1.06의 확률을 평균합니다. 현재 PC의 완성된 family cache가 있으면 재사용하고, 다른 PC에서는 checkpoint로부터 생성합니다.

In [ ]:
from evaluate_resnet50_tta_ensemble import predict_tta
from traffic_signs.runtime import load_checkpoint

master_names = np.asarray([s.filename for s in master_samples])
master_labels = np.asarray([s.label for s in master_samples], dtype=np.int64)
test_names = np.asarray([s.filename for s in test_samples])
master_index = {name: i for i, name in enumerate(master_names.tolist())}
cache_dir = FINAL_WORK / 'tta_fold_cache'
cache_dir.mkdir(exist_ok=True)

precomputed = {
 'resnet80': (PROJECT_ROOT/'outputs/experiments/resnet50_interpolated_80_112_5fold/size_80/tta_oof_probabilities.npz', PROJECT_ROOT/'outputs/experiments/resnet50_interpolated_80_112_5fold/size_80/tta_test_probabilities.npz'),
 'resnet96': (PROJECT_ROOT/'outputs/experiments/resnet50_finetune_5fold/size_96/tta_oof_probabilities.npz', PROJECT_ROOT/'outputs/experiments/resnet50_finetune_5fold/size_96/tta_test_probabilities.npz'),
 'resnet128': (PROJECT_ROOT/'outputs/experiments/resnet50_finetune_5fold/size_128/tta_oof_probabilities.npz', PROJECT_ROOT/'outputs/experiments/resnet50_finetune_5fold/size_128/tta_test_probabilities.npz'),
 'convnext160': (PROJECT_ROOT/'outputs/experiments/gtsrb_improvement_20260805/convnext160_5fold_tta_oof_probabilities.npz', PROJECT_ROOT/'outputs/experiments/gtsrb_improvement_20260805/convnext160_5fold_tta_test_probabilities.npz')
}

def load_npz(path):
    with np.load(path) as d: return {k: d[k] for k in d.files}

def aggregate_family(name, specs):
    old_oof, old_test = precomputed[name]
    if old_oof.exists() and old_test.exists():
        a, b = load_npz(old_oof), load_npz(old_test)
        assert np.array_equal(a['filenames'], master_names) and np.array_equal(b['filenames'], test_names)
        print('reuse family TTA:', name)
        return a['probabilities'].astype(np.float64), b['probabilities'].astype(np.float64)
    oof = np.full((len(master_names), 43), np.nan, dtype=np.float64)
    test_sum = np.zeros((len(test_names), 43), dtype=np.float64)
    for spec in specs:
        tag = f"{name}_fold{spec['fold']}"
        val_cache, test_cache = cache_dir/f'{tag}_oof.npz', cache_dir/f'{tag}_test.npz'
        if val_cache.exists() and test_cache.exists():
            val, tst = load_npz(val_cache), load_npz(test_cache)
        else:
            model, cfg, _ = load_checkpoint(spec['dir']/'best_model.pt', torch.device(DEVICE))
            val_samples = load_split_manifest(spec['dir']/'split_manifest.csv', 'val')
            val = predict_tta(model, cfg, val_samples, torch.device(DEVICE), 32, 0)
            tst = predict_tta(model, cfg, test_samples, torch.device(DEVICE), 32, 0)
            np.savez_compressed(val_cache, **val); np.savez_compressed(test_cache, **tst)
            del model; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        positions = np.asarray([master_index[str(x)] for x in val['filenames']], dtype=np.int64)
        assert np.array_equal(master_labels[positions], val['labels'])
        oof[positions] = val['probabilities']
        assert np.array_equal(test_names, tst['filenames'])
        test_sum += tst['probabilities']
    assert np.isfinite(oof).all()
    return oof, test_sum / len(specs)

family_probabilities = {name: aggregate_family(name, specs) for name, specs in families.items()}


### Ensemble and Evaluation
ResNet 내부 weight는 80/96/128px = 0.1/0.4/0.5이고, 최종 family weight는 ResNet/ConvNeXt = 0.6/0.4입니다. 이 weight는 TEST가 아닌 OOF에서 고정했습니다.

In [ ]:
r80_oof, r80_test = family_probabilities['resnet80']
r96_oof, r96_test = family_probabilities['resnet96']
r128_oof, r128_test = family_probabilities['resnet128']
cx_oof, cx_test = family_probabilities['convnext160']
resnet_oof = 0.1*r80_oof + 0.4*r96_oof + 0.5*r128_oof
resnet_test = 0.1*r80_test + 0.4*r96_test + 0.5*r128_test
final_oof = 0.6*resnet_oof + 0.4*cx_oof
final_test = 0.6*resnet_test + 0.4*cx_test
final_oof /= final_oof.sum(axis=1, keepdims=True)
final_test /= final_test.sum(axis=1, keepdims=True)
oof_pred = final_oof.argmax(1)
metrics = {'macro_f1': f1_score(master_labels, oof_pred, average='macro'),
           'accuracy': accuracy_score(master_labels, oof_pred),
           'nll': log_loss(master_labels, final_oof, labels=list(range(43))),
           'errors': int((oof_pred != master_labels).sum())}
print(json.dumps(metrics, indent=2))
assert metrics['macro_f1'] > 0.99


### Save Results to CSV
원본 `result.csv`의 ID 순서와 기존 header 유무를 그대로 보존하며 별도 index를 추가하지 않습니다.

In [ ]:
template_rows = list(csv.reader(TEMPLATE.open(newline='', encoding='utf-8-sig')))
has_header = bool(template_rows and template_rows[0] and template_rows[0][0].lower() in {'id','filename','file'})
header = template_rows[0] if has_header else None
body = template_rows[1:] if has_header else template_rows
template_ids = [row[0] for row in body]
assert len(template_ids) == 8670 and len(set(template_ids)) == 8670
assert set(template_ids) == set(test_names.tolist())
prediction_by_id = {name: int(pred) for name, pred in zip(test_names.tolist(), final_test.argmax(1).tolist())}
with FINAL_CSV.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.writer(handle)
    if header is not None: writer.writerow(header)
    for row in body:
        writer.writerow([row[0], prediction_by_id[row[0]]])
check_rows = list(csv.reader(FINAL_CSV.open(newline='', encoding='utf-8-sig')))
check_body = check_rows[1:] if has_header else check_rows
assert [r[0] for r in check_body] == template_ids
assert all(len(r)==2 and r[1].isdigit() and 0 <= int(r[1]) < 43 for r in check_body)
print('saved:', FINAL_CSV, 'rows:', len(check_body), 'header_preserved:', has_header)


## 최종 확인
생성 파일: `DS2_challenge_team1_final.csv`

TEST class 분포나 reference label은 모델 선택 및 weight tuning에 사용하지 않았습니다. 제출 전 `Restart Kernel and Run All Cells`로 재실행하십시오.